In [ ]:
# pyright: reportGeneralTypeIssues=false, reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false
# ruff: noqa
# pylint: skip-file

# NHANES Diabetes Prediction — Random Forest Baseline

Untuned Random Forest using LightGBM RF mode (`boosting_type='rf'`).
Default parameters, no feature engineering, no hyperparameter optimization.
Purpose: establish a baseline for comparison with tuned LGBM and ensemble methods.

In [ ]:
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

In [ ]:
WANDB_PROJECT = "Model exploration for Diabetes Prediction"
ENTITY = "fastegiano-tesis"
STUDY_NAME = "rf_baseline_20172021"
RANDOM_STATE = 37

USE_CALIBRATION = False
CAT_FEATURES = [
    "education_level",
    "has_partner",
    "had_partner",
    "is_female",
    "ever_smoker",
]

In [ ]:
data = pd.read_csv("../../../dataset/processed_data_combined_2017_2021.csv")
data.head()

In [ ]:
X = data.drop(columns=["has_diabetes_or_prediabetes", "cycle", "survey_weight"], errors="ignore")
y = data["has_diabetes_or_prediabetes"]

print(f"Features: {X.shape[1]}")
print(f"Feature names: {list(X.columns)}")

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15, stratify=y_trainval, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Prevalence — Train: {y_train.mean():.1%}, Val: {y_val.mean():.1%}, Test: {y_test.mean():.1%}")

In [ ]:
rf_params = {
    "boosting_type": "rf",
    "objective": "binary",
    "metric": "binary_logloss",
    "verbosity": -1,
    "n_estimators": 1000,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "feature_fraction": 0.8,
    "random_state": RANDOM_STATE,
}

model = lgb.LGBMClassifier(**rf_params)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_names=["train", "valid"],
    callbacks=[lgb.early_stopping(50, verbose=False)],
    categorical_feature=CAT_FEATURES,
)

In [ ]:
y_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)[:, 1]

recall_pos = recall_score(y_test, y_pred, pos_label=1)
recall_neg = recall_score(y_test, y_pred, pos_label=0)
print(f"Recall (positive): {recall_pos:.2%}")
print(f"Recall (negative): {recall_neg:.2%}")
print(f"y_test distribution:\n{y_test.value_counts()}")
print(f"y_pred distribution:\n{pd.Series(y_pred).value_counts()}")

In [ ]:
results = model.evals_result_

fig_overfit, ax = plt.subplots(figsize=(10, 6))
ax.plot(results["train"]["binary_logloss"], label="Train", linewidth=2)
ax.plot(results["valid"]["binary_logloss"], label="Valid", linewidth=2)
ax.set_xlabel("Iteration")
ax.set_ylabel("Binary Log Loss")
ax.set_title("Binary Log Loss Over Training Iterations (RF)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig_cm_05, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Diabetes", "Diabetes"]).plot(cmap="Blues", ax=ax)
ax.set_title(f"Confusion Matrix — Threshold = 0.50\nRecall: {recall_pos:.2%}")
plt.tight_layout()
plt.show()

In [ ]:
fi_sorted = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)

fig_fi_splits, ax = plt.subplots(figsize=(10, 8))
fi_sorted.plot(kind="barh", ax=ax)
ax.set_title("Feature Importances — Splits (RF)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
fi_gain = pd.Series(
    model.booster_.feature_importance(importance_type="gain"),
    index=X.columns
).sort_values(ascending=True)

fig_fi_gain, ax = plt.subplots(figsize=(10, 8))
fi_gain.plot(kind="barh", ax=ax)
ax.set_title("Feature Importances — Gain (RF)")
ax.set_xlabel("Gain")
plt.tight_layout()
plt.show()

In [ ]:
# Threshold optimization
y_proba = model.predict_proba(X_val)[:, 1]

# Calculate PR curve
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
pr_auc = auc(recalls, precisions)

# Create figure with 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Precision-Recall Curve ---
ax1 = axes[0]
ax1.plot(recalls, precisions, "b-", linewidth=2, label=f"PR Curve (AUC={pr_auc:.3f})")
ax1.fill_between(recalls, precisions, alpha=0.2)

# Mark key threshold points
target_recalls = [0.80, 0.75, 0.70, 0.50]
colors = ["red", "orange", "green", "purple"]

for target, color in zip(target_recalls, colors):
    idx = np.where(recalls[:-1] >= target)[0]
    if len(idx) > 0:
        i = idx[-1]
        thresh = thresholds[i]
        ax1.scatter(
            recalls[i],
            precisions[i],
            c=color,
            s=100,
            zorder=5,
            label=f"Recall={target:.0%} (thresh={thresh:.3f}, prec={precisions[i]:.1%})",
        )

# Baseline (random classifier)
baseline = y_val.mean()
ax1.axhline(
    y=baseline,
    color="gray",
    linestyle="--",
    label=f"Baseline (prevalence={baseline:.1%})",
)

ax1.set_xlabel("Recall (Sensitivity)", fontsize=12)
ax1.set_ylabel("Precision (PPV)", fontsize=12)
ax1.set_title("Precision-Recall Curve (RF)", fontsize=14)
ax1.legend(loc="upper right", fontsize=9)
ax1.set_xlim([0, 1.02])
ax1.set_ylim([0, 1.02])
ax1.grid(True, alpha=0.3)

# --- Plot 2: Threshold vs Metrics ---
ax2 = axes[1]

thresh_range = np.linspace(0.05, 0.6, 100)
recall_at_thresh = []
precision_at_thresh = []
f1_at_thresh = []
flagged_pct = []

for t in thresh_range:
    y_pred_t = (y_proba >= t).astype(int)
    r = recall_score(y_val, y_pred_t, zero_division=0)
    p = precision_score(y_val, y_pred_t, zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    recall_at_thresh.append(r)
    precision_at_thresh.append(p)
    f1_at_thresh.append(f1)
    flagged_pct.append(y_pred_t.mean())

ax2.plot(thresh_range, recall_at_thresh, "b-", linewidth=2, label="Recall")
ax2.plot(thresh_range, precision_at_thresh, "r-", linewidth=2, label="Precision")
ax2.plot(thresh_range, f1_at_thresh, "g--", linewidth=2, label="F1 Score")
ax2.plot(thresh_range, flagged_pct, "k:", linewidth=2, label="% Flagged")

# Mark default 0.5 threshold
ax2.axvline(x=0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")

# Mark optimal threshold for 80% recall
target_80_idx = np.argmin(np.abs(np.array(recall_at_thresh) - 0.80))
optimal_thresh = thresh_range[target_80_idx]
ax2.axvline(
    x=optimal_thresh,
    color="red",
    linestyle="--",
    alpha=0.7,
    label=f"80% Recall (thresh={optimal_thresh:.3f})",
)

ax2.set_xlabel("Decision Threshold", fontsize=12)
ax2.set_ylabel("Score", fontsize=12)
ax2.set_title("Metrics vs Decision Threshold (RF)", fontsize=14)
ax2.legend(loc="center right", fontsize=9)
ax2.set_xlim([0.05, 0.6])
ax2.set_ylim([0, 1.02])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Print Summary Table ---
print("\n" + "=" * 70)
print("THRESHOLD ANALYSIS SUMMARY (RF)")
print("=" * 70)
print(f"{'Threshold':<12} {'Recall':<12} {'Precision':<12} {'F1':<12} {'Flagged':<12}")
print("-" * 70)

for thresh in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    y_pred_t = (y_proba >= thresh).astype(int)
    r = recall_score(y_val, y_pred_t, zero_division=0)
    p = precision_score(y_val, y_pred_t, zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    n_flagged = y_pred_t.sum()
    pct_flagged = y_pred_t.mean() * 100
    print(
        f"{thresh:<12.2f} {r:<12.1%} {p:<12.1%} {f1:<12.3f} {n_flagged} ({pct_flagged:.1f}%)"
    )

print("=" * 70)
print(
    f"\nValidation set: {len(y_val)} samples, {y_val.sum()} diabetes cases ({y_val.mean():.1%} prevalence)"
)

In [ ]:
y_test_proba = model.predict_proba(X_test)[:, 1]
y_pred_final = (y_test_proba >= optimal_thresh).astype(int)

r = recall_score(y_test, y_pred_final, zero_division=0)
p = precision_score(y_test, y_pred_final, zero_division=0)
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

print(f"\nFinal chosen threshold: {optimal_thresh}")
print(f" - Recall: {r:.2%}")
print(f" - Precision: {p:.2%}")
print(f" - F1 Score: {f1:.3f}")

In [ ]:
cm_final = confusion_matrix(y_test, y_pred_final)
fig_cm_opt, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm_final, display_labels=["No Diabetes", "Diabetes"]).plot(cmap="Blues", ax=ax)
ax.set_title(f"Confusion Matrix — Threshold = {optimal_thresh:.3f}\n(recall={r:.2%}, precision={p:.2%})")
plt.tight_layout()
plt.show()

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT,
    entity=ENTITY,
    name=f"{STUDY_NAME}-evaluation",
    job_type="evaluation",
    config={
        **rf_params,
        "chosen_threshold": optimal_thresh,
        "use_calibration": USE_CALIBRATION,
        "model_type": "random_forest_lgbm",
        "tuned": False,
    },
)

In [ ]:
y_probas_both = model.predict_proba(X_test)
wandb.log({
    "pr_curve": wandb.plot.pr_curve(
        y_true=y_test.values,
        y_probas=y_probas_both,
        labels=["No Diabetes", "Diabetes"],
    )
})

In [ ]:
wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        y_true=y_test.values,
        preds=y_pred_final,
        class_names=["No Diabetes", "Diabetes"],
    )
})

In [ ]:
y_val_proba = model.predict_proba(X_val)[:, 1]
threshold_data = []
for t in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    y_pred_t = (y_val_proba >= t).astype(int)
    threshold_data.append([
        t,
        recall_score(y_val, y_pred_t, zero_division=0),
        precision_score(y_val, y_pred_t, zero_division=0),
        f1_score(y_val, y_pred_t, zero_division=0),
        y_pred_t.mean(),
    ])

wandb.log({
    "threshold_analysis": wandb.Table(
        columns=["Threshold", "Recall", "Precision", "F1", "Pct_Flagged"],
        data=threshold_data,
    )
})

In [ ]:
import os

chart_dir = "wandb_charts"
os.makedirs(chart_dir, exist_ok=True)

charts = {
    "confusion_matrix_050": fig_cm_05,
    "confusion_matrix_optimal": fig_cm_opt,
    "overfitting_curve": fig_overfit,
    "pr_threshold_analysis": fig,
    "feature_importance_splits": fig_fi_splits,
    "feature_importance_gain": fig_fi_gain,
}

for name, figure in charts.items():
    figure.savefig(f"{chart_dir}/{name}.png", dpi=150, bbox_inches="tight")

artifact = wandb.Artifact(
    name=f"{STUDY_NAME}-charts",
    type="evaluation-charts",
    description="All evaluation charts for RF baseline model",
    metadata={
        "threshold_default": 0.5,
        "threshold_optimal": optimal_thresh,
        "test_recall": r,
        "test_precision": p,
    },
)
artifact.add_dir(chart_dir)
wandb.log_artifact(artifact)

wandb.log({name: wandb.Image(figure) for name, figure in charts.items()})

In [ ]:
y_pred_05 = (y_test_proba >= 0.5).astype(int)

wandb.summary.update({
    "test_recall": r,
    "test_precision": p,
    "test_f1": f1,
    "test_ap": average_precision_score(y_test, y_test_proba),
    "test_recall_at_050": recall_score(y_test, y_pred_05),
    "test_precision_at_050": precision_score(y_test, y_pred_05),
    "chosen_threshold": optimal_thresh,
    "n_iterations_used": model.best_iteration_,
    "use_calibration": USE_CALIBRATION,
    "test_size": len(y_test),
    "val_size": len(y_val),
    "train_size": len(y_train),
    "test_prevalence": y_test.mean(),
    "n_features": X.shape[1],
    "model_type": "random_forest_lgbm",
    "tuned": False,
})

wandb.finish()